In [ ]:
# Importing required libs
from bs4 import BeautifulSoup
import pandas as pd
import requests


# Creating global variables
log_file = "./etl_project_log.txt"
url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'

table_attribs = ["Country", "GDP_USD_millions"]
countries_row = []
imf_estimate_row = []
soup = BeautifulSoup(requests.get(url).text,"html5lib")
table = soup.find_all("table",class_="wikitable")[0]
caption = table.find("caption").text
tables = table.find_all("tbody")[0]
for rows in tables.find_all("tr"):
    countries = rows.find_all("a")
    for index,country in enumerate(countries):
        if index==0:
            countries_row.append(country.getText(strip=True))

    currency_values = rows.find_all("td")
    for index,vals in enumerate(currency_values):
        if index==2:
            imf_estimate_row.append(vals.get_text(strip=True))

       
df = pd.DataFrame(list(zip(countries_row,imf_estimate_row)),columns=table_attribs).drop(index=0).reset_index(drop=True)
df

,Country,GDP_USD_millions
0,United States,"26,854,599"
1,China,"19,373,586"
2,Japan,"4,409,738"
3,Germany,"4,308,854"
4,India,"3,736,882"
...,...,...
208,Anguilla,—
209,Kiribati,248
210,Nauru,151
211,Montserrat,—


In [ ]:
# headers = []
# lower_headers = []

# # Scraping the webpage
# soup = BeautifulSoup(requests.get(url).content,"html5lib")
# table = soup.find_all("table",class_="wikitable")[0]
# caption = table.find("caption").text
# for index,cols in enumerate(table.find_all("tr")[:2]):
#     if index==0:
#         for i in range(len(cols.find_all("th"))):
#             headers.append(str(cols.find_all("th")[i].contents[0].text).strip("\n"))
#     else:
#         for i in range(len(cols.find_all("th"))):
#             lower_headers.append(str(cols.find_all("th")[i].contents[0].text).strip("\n"))
                    
# # Define MultiIndex for columns
# # Initialize an empty list to hold the tuples
# tuples = []

# # Loop through headers and lower_headers
# for header in headers:
#     if header in ['IMF', 'World Bank', 'United Nations']:
#         for lower_header in set(lower_headers):
#             tuples.append((header, lower_header))
#     else:
#         tuples.append((header,""))

# # Create MultiIndex from the list of tuples
# columns = pd.MultiIndex.from_tuples(tuples)

In [ ]:
# Creating global variables
log_file = "./etl_project_log.txt"
db_name = 'World_Economies.db'
table_name = 'Countries_by_GDP'
csv_path = './Countries_by_GDP.csv'

def extract(url,table_attribs):
    extract = requests.get(url).text
    soup_parsed = BeautifulSoup(extract,"html.parser")
    df = pd.DataFrame(columns=table_attribs)
    t_body_attrs = soup_parsed.find_all("tbody")[2]
    for rows in t_body_attrs.find_all("tr"):
        col = rows.find_all('td')
        if len(col)!=0:
            if col[0].find("a") is not None and "—" not in col[2]:
                data_dict = {table_attribs[0]:col[0].a.get_text(strip=True),
                             table_attribs[1]:col[2].get_text(strip=True)}
                df1 = pd.DataFrame(data_dict,index=[0])
                df = pd.concat([df,df1],ignore_index=True)
    return df

In [106]:
url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'
table_attribs = ["Country", "GDP_USD_millions"]
df = extract(url,table_attribs)
df["GDP_USD_millions"] = df["GDP_USD_millions"].apply(lambda x: round(float(x.replace(",",""))*0.001,2))
df = df.rename(columns={"GDP_USD_millions":"GDP_USD_billions"})
df

,Country,GDP_USD_billions
0,United States,26854.60
1,China,19373.59
2,Japan,4409.74
3,Germany,4308.85
4,India,3736.88
...,...,...
186,Marshall Islands,0.29
187,Palau,0.26
188,Kiribati,0.25
189,Nauru,0.15
